# Compare Base Round Robin, ARRTQ, and modified ARRTQ

In [14]:
import sys, os
sys.path.append(os.path.abspath(".."))  # add parent directory to search path

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from core.process import Process
from core.scheduler_arrtq import arrtq
from core.scheduler_rr import round_robin
from core.scheduler_modified_ARRTQ import arrtq_balanced
from core.process_generator import generate_processes
import plotly.figure_factory as ff
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

In [17]:
datasets = {
    "20_processes": generate_processes(20, seed=1),
    "50_processes": generate_processes(50, seed=2),
    "80_processes": generate_processes(80, seed=3),
    "100_processes": generate_processes(100, seed=4),
    "1000_processes": generate_processes(1000, seed=5)
}

In [18]:
def show_processes(processes):
    df = pd.DataFrame([{
        'PID': p.pid,
        'Arrival Time': p.arrival_time,
        'Burst Time': p.burst_time
    } for p in processes])
    display(df.head(10))
    print(f"Total processes: {len(processes)}")

show_processes(datasets["1000_processes"])

,PID,Arrival Time,Burst Time
0,1,2,13
1,2,6,18
2,3,6,16
3,4,10,9
4,5,13,7
5,6,13,17
6,7,19,14
7,8,21,20
8,9,21,8
9,10,22,7


Total processes: 1000


In [20]:
context_switch_time = 1
processes = datasets["1000_processes"]

import copy
arrtq_metrics = arrtq(copy.deepcopy(processes), context_switch_time)
arrtq_balanced_metrics = arrtq_balanced(copy.deepcopy(processes), context_switch_time, c=0.9)


def build_results_df(metrics):
    return pd.DataFrame([{
        'Process ID': p.pid,
        'Arrival Time': p.arrival_time,
        'Burst Time': p.burst_time,
        'Completion Time': p.completion_time,
        'Turnaround Time': p.turnaround_time,
        'Waiting Time': p.waiting_time,
        'Start Time': p.start_time,
        'First Response Time': metrics["first_response_times"].get(p.pid, None),
    } for p in metrics['completed_processes']
    
    ])

results_arrtq = build_results_df(arrtq_metrics)
results_balanced = build_results_df(arrtq_balanced_metrics)

In [21]:
compare_df = pd.DataFrame({
    'Metric': [
        'Average Turnaround Time', 'Average Waiting Time',
        'Average First Response Time', 'Context Switches',
        'CPU Utilization (%)', 'Throughput'
    ],
    'ARRTQ': [
        arrtq_metrics['average_turnaround_time'],
        arrtq_metrics['average_waiting_time'],
        arrtq_metrics['average_first_response_time'],
        arrtq_metrics['context_switches'],
        arrtq_metrics['cpu_utilization'],
        arrtq_metrics['throughput']
    ],
    'ARRTQ_Balanced': [
        arrtq_balanced_metrics['average_turnaround_time'],
        arrtq_balanced_metrics['average_waiting_time'],
        arrtq_balanced_metrics['average_first_response_time'],
        arrtq_balanced_metrics['context_switches'],
        arrtq_balanced_metrics['cpu_utilization'],
        arrtq_balanced_metrics['throughput']
    ]
})
display(compare_df)

# --- Bar chart for metric comparison ---
fig = go.Figure()
for col, color in zip(['ARRTQ', 'ARRTQ_Balanced'], ['steelblue', 'seagreen']):
    fig.add_trace(go.Bar(
        x=compare_df['Metric'],
        y=compare_df[col],
        name=col,
        marker_color=color
    ))

fig.update_layout(
    title='Performance Comparison: ARRTQ vs ARRTQ-Balanced',
    xaxis_title='Metric',
    yaxis_title='Value',
    barmode='group'
)
fig.show()

# --- Plot First Response Time comparison for each process ---
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    x=results_arrtq['Process ID'], y=results_arrtq['First Response Time'],
    mode='lines+markers', name='ARRTQ', line=dict(color='blue')
))
fig2.add_trace(go.Scatter(
    x=results_balanced['Process ID'], y=results_balanced['First Response Time'],
    mode='lines+markers', name='ARRTQ-Balanced', line=dict(color='green')
))
fig2.update_layout(
    title='First Response Time per Process',
    xaxis_title='Process ID',
    yaxis_title='Time Units'
)
fig2.show()

# --- Optional: Ready Queue length over time ---
rq_df_balanced = pd.DataFrame(arrtq_balanced_metrics['rq_length_over_time'], columns=['Time', 'RQ Length'])
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    x=rq_df_balanced['Time'], y=rq_df_balanced['RQ Length'],
    mode='lines+markers', name='ARRTQ-Balanced', line=dict(color='green')
))
fig3.add_trace(go.Scatter(
    x=pd.DataFrame(arrtq_metrics['rq_length_over_time'], columns=['Time', 'RQ Length'])['Time'],
    y=pd.DataFrame(arrtq_metrics['rq_length_over_time'], columns=['Time', 'RQ Length'])['RQ Length'],
    mode='lines+markers', name='ARRTQ', line=dict(color='orange')
))
fig3.update_layout(
    title='Ready Queue Length Over Time (ARRTQ vs Balanced)',
    xaxis_title='Time Units',
    yaxis_title='Queue Length'
)
fig3.show()

,Metric,ARRTQ,ARRTQ_Balanced
0,Average Turnaround Time,5626.564495,5831.567096
1,Average Waiting Time,5615.427495,5820.430096
2,Average First Response Time,4284.699806,3295.039785
3,Context Switches,2124.000000,2078.000000
4,CPU Utilization (%),99.984918,99.984866
5,Throughput,0.075409,0.075672
